In [1]:
import torch
from tqdm.notebook import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM
from torch.nn import CrossEntropyLoss
import re
import os
import gc
import pandas as pd
from bs4 import BeautifulSoup,MarkupResemblesLocatorWarning
import warnings
import torch
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset
from tqdm import tqdm


In [2]:

ROOT = "" 
# directories             
DATA_DIR = os.path.join(ROOT, "Data")
RESULTS_DIR = os.path.join(ROOT, "results")

# subjects and models
SUBJECTS = ['stackoverflow']
LLMS = ["meta-llama/Llama-3.1-8B"]

# batch/hyperparams
BATCH_SIZE = 16
MAX_LENGTH = 256
device = "cuda" if torch.cuda.is_available() else "cpu"

In [3]:


warnings.filterwarnings("ignore", category=MarkupResemblesLocatorWarning)

def preprocess_text(df: pd.DataFrame) -> pd.DataFrame:
    """
    Remove HTML tags from 'title' and 'body' columns of the dataframe.

    Args:
        df (pd.DataFrame): Input dataframe with 'title' and 'body' columns.

    Returns:
        pd.DataFrame: Cleaned dataframe with HTML removed.
    """
    df = df.copy().head(1000)  # Process only the first 1000 rows for efficiency

    for col in ['Title', 'Body', 'title', 'body']:
        if col in df.columns:
            df[col] = df[col].astype(str).apply(lambda x: BeautifulSoup(x, "html.parser").get_text())
    if 'Title' in df.columns and 'Body' in df.columns:
        df['text'] = df['Title'].fillna('') + '\n' + df['Body'].fillna('')        
    else:
        df['text'] = df['title'].fillna('') + '\n' + df['body'].fillna('')
    return df



In [4]:

def process_dataset(texts, model, tokenizer, batch_size=32, max_length=64, device="cuda", num_workers=2):
    """
    Compute perplexities for a dataset of texts using DataLoader for efficiency.

    Args:
        texts (List[str]): Input texts.
        model: Hugging Face causal LM model.
        tokenizer: Hugging Face tokenizer.
        batch_size (int): Number of texts per batch.
        max_length (int): Max sequence length for truncation.
        device (str): Device ("cuda" or "cpu").
        num_workers (int): DataLoader workers (0 = no multiprocessing).

    Returns:
        List[float | None]: Perplexity per text (None if failed).
    """
    # 1. Tokenize once
    encodings = tokenizer(
        texts,
        return_tensors="pt",
        padding=True,
        truncation=True,
        max_length=max_length
    )

    dataset = TensorDataset(encodings["input_ids"], encodings["attention_mask"])
    loader = DataLoader(dataset, batch_size=batch_size, num_workers=num_workers)

    perps = []
    # 2. Iterate over DataLoader
    for batch_idx, (input_ids, attention_mask) in tqdm(enumerate(tqdm(loader))):
        input_ids, attention_mask = input_ids.to(device), attention_mask.to(device)

        try:
            with torch.no_grad():
                outputs = model(input_ids, attention_mask=attention_mask)
                logits = outputs.logits

                # Shift for causal LM loss
                shift_logits = logits[:, :-1, :]
                shift_labels = input_ids[:, 1:]
                shift_mask = attention_mask[:, 1:]

                # log-softmax for efficiency
                log_probs = F.log_softmax(shift_logits, dim=-1)
                token_log_probs = log_probs.gather(-1, shift_labels.unsqueeze(-1)).squeeze(-1)

                # Mask padding tokens
                token_log_probs = token_log_probs * shift_mask

                # Average per sequence
                avg_log_probs = token_log_probs.sum(dim=1) / shift_mask.sum(dim=1)
                batch_perps = torch.exp(-avg_log_probs).tolist()

            perps.extend(batch_perps)

        except Exception as e:
            print(f"→ batch {batch_idx} failed: {e}")
            perps.extend([None] * input_ids.size(0))

    return perps


### In the following section we are going to compute the Perplexity for each question in our dataset. We have to run the previous cells in order to do so.
The results will be saved in the RESULTS_DIR 

In [ ]:
for model_name in LLMS:
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
    slug = model_name.replace("/", "-")
    print(f"\n=== Loading and running {model_name} → folder `{slug}` ===")

    print(f"→ Loading tokenizer for: {model_name}")
    tok = AutoTokenizer.from_pretrained(model_name)
    tok.pad_token = tok.eos_token

    print(f"→ Loading model for: {model_name}")
    mdl = AutoModelForCausalLM.from_pretrained(model_name)
    mdl.eval().to(device)

    for subj in SUBJECTS:
        subject_dir = os.path.join(DATA_DIR, subj)
        pq_files = [f for f in os.listdir(subject_dir) if f.endswith(".pq")]
        print(f"\n→ Subject: {subj} | Found {len(pq_files)} Parquet files.")

        for pq_file in pq_files:
            print(f"\n→ Processing file: {pq_file}")
            inp_path = os.path.join(subject_dir, pq_file)

            print("→ Reading Parquet file...")
            df = pd.read_parquet(inp_path).head(1000)  # Process only the first 1000 rows for efficiency

            print("→ Preprocessing Title and Body into 'text' column...")
            df = preprocess_text(df)

            print("→ Calculating perplexity for each row...")
            txt_col = df["text"].astype(str).tolist()
            perps = process_dataset(txt_col, mdl, tok, batch_size=BATCH_SIZE, max_length=MAX_LENGTH, device=device)

            out_dir = os.path.join(RESULTS_DIR, subj, pq_file.replace(".pq", ""), slug)
            os.makedirs(out_dir, exist_ok=True)
            out_file = os.path.join(out_dir, pq_file.replace(".pq", "_with_perplexity_256.parquet"))

            print(f"→ Saving results to: {out_file}")
            df["perplexity"] = perps
            df.to_parquet(out_file, index=False)
            print(df["perplexity"].describe())
            print(f"✔ Done with {subj}/{pq_file} → {slug}")
